# FT09 — VLM baseline evaluation
 Evaluate the pretrained model before FT09 fine-tuning. Do not run any training in this notebook.
 Game, prompt, parser and logging are shared with the other notebooks.

 **Start with a GPU in JupyterHub.**
 Begin with the 2B model and three turns. Switch models by rerunning the settings and model-loading cells; keep the new model's baseline separate.


## Dependencies
Run this cell first. If packages were installed or upgraded, restart the kernel, then continue with the cells below.


In [ ]:
%pip install -q "arc-agi==0.9.9" numpy pillow matplotlib pyarrow
%pip install -q -U transformers accelerate


In [ ]:
from pathlib import Path
import sys

# Open this notebook from the project folder containing ft09_shared.py.
PROJECT_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "ft09_shared.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise RuntimeError("Open the notebook from the FT09 project folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import ft09_shared as ft
ft.configure_project(PROJECT_DIR)
print("Project:", PROJECT_DIR)


## 1. Settings
 For the first connection check, leave the defaults below.
 For the official baseline, choose the final model, set `PHASE = "baseline"`, `LEVELS = [1, 2, 3, 4, 5]`, `N_ATTEMPTS = 5` and `MAX_STEPS = None`.
 Use the same fixed settings before and after fine-tuning. Check reply truncation before freezing the output-token budget.


In [ ]:
MODEL = "Qwen/Qwen3.5-2B"  # Later: "Qwen/Qwen3.5-9B"
PHASE = "connection_check"
LEVELS = [1]
N_ATTEMPTS = 1
TEMPERATURE = 0.7
MAX_STEPS = 3              # None uses the agreed per-level limits.
MAX_NEW_TOKENS = 512
SHOW_FRAMES = True
PROMPT = ft.GAME_PROMPT


## 2. Load the model once
 Rerun this cell only when changing model/loading settings or after restarting the kernel.
 Imports do not download models; this cell does. It uses a single CUDA GPU and disables extended thinking mode while still asking for the short REASONING field.


In [ ]:
from ft09_models import LocalQwenBackend

if "backend" in globals():
    backend.close()
backend = LocalQwenBackend(MODEL, max_new_tokens=MAX_NEW_TOKENS)
print(backend.info())


## 3. Start a new run directory
 Run this cell for each new evaluation batch. It keeps models/batches separate, writes configuration and enables request logs.
 Results go under `evaluation/`; training files are not modified.


In [ ]:
if backend.model_name != MODEL or backend.max_new_tokens != MAX_NEW_TOKENS:
    raise RuntimeError("Reload the model after changing MODEL or MAX_NEW_TOKENS.")
RUN_DIR = ft.new_run_dir("evaluation", MODEL)
backend.log_path = RUN_DIR / "requests.jsonl"
limits = {
    level: ft.step_limit(level, ft.get_baseline_actions())
    if MAX_STEPS is None else MAX_STEPS
    for level in LEVELS
}
ft.save_run_config(
    RUN_DIR, backend, PROMPT, phase=PHASE, levels=LEVELS,
    attempts=N_ATTEMPTS, temperature=TEMPERATURE, step_limits=limits,
)
print("Output:", RUN_DIR)
print("Step limits:", limits)


## 4. Run the selected levels
 Every attempt resets the board and clears memory. Records are saved after every turn.
 Failed and invalid turns stay in the logs. Existing attempt files are never overwritten.
 The limits count decision turns; executed clicks are counted separately.


In [ ]:
all_records = []
for level in LEVELS:
    all_records.extend(ft.run_level(
        backend, level, RUN_DIR,
        n_attempts=N_ATTEMPTS, temperature=TEMPERATURE,
        max_steps=MAX_STEPS, prompt=PROMPT, show_frames=SHOW_FRAMES,
    ))


## 5. Results
 Creates per-attempt `results.csv` and per-level `summary.csv`. Technical failures and unavailable token usage are reported rather than silently treated as successful runs or zero-token answers.


In [ ]:
attempt_results, level_summary = ft.summarise_evaluation(RUN_DIR)


## 6. Inspect a saved attempt


In [ ]:
FILE = next(iter(sorted(RUN_DIR.glob("ft09_level_*_attempt_*.parquet"))), None)
if FILE is not None:
    ft.review_steps(FILE, first_step=1, last_step=3)


## Held-out Level 6
 Keep Level 6 outside prompt development and training. Once the configuration is fixed, run its evaluation as a separate batch and report it separately.
 Do not add these runs to the training export.

 After changing shared Python files, restart the kernel before rerunning setup. You do not need to reload the model for a new level or attempt batch.
